# OW-OVD Visualization Notebook (Task 1)
Notebook trực quan hóa mô hình OW-OVD sử dụng dữ liệu IP102 mới và checkpoint Task 1.

In [ ]:
# Cell 1: Clone repository và thiết lập thư mục làm việc trên Kaggle
import os

repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("Cloning repository...")
    !git clone {repo_url} {working_dir}
else:
    print("Repository đã tồn tại. Đang cập nhật (pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

if not os.path.exists("third_party/mmyolo"):
    print("Cloning mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("mmyolo đã tồn tại.")


In [ ]:
# Cell 2: Cài đặt toàn bộ môi trường và các thư viện cần thiết
!pip install torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html
!pip install matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers
!pip install "mmdet>=3.1.0" --no-deps
!pip install --no-build-isolation --no-deps third_party/mmyolo


In [ ]:
# Cell 3: Vá lỗi giới hạn phiên bản MMCV trong mmdet và mmyolo
import importlib.util
import sys

packages = ['mmdet', 'mmyolo']
for pkg in packages:
    spec = importlib.util.find_spec(pkg)
    if spec and spec.origin:
        try:
            with open(spec.origin, 'r') as f:
                content = f.read()
            new_content = content.replace('2.1.0', '2.3.0').replace('2.2.0', '2.3.0')
            with open(spec.origin, 'w') as f:
                f.write(new_content)
            print(f"-> Đã vá lỗi phiên bản MMCV cho {pkg} thành công.")
        except Exception as e:
            print(f"-> Lỗi khi vá {pkg}: {e}")


In [ ]:
# Cell 4: Tự động định vị đường dẫn dataset IP102 mới & Checkpoint Task 1
import os
import glob

dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break

if dataset_root is None:
    paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

if dataset_root is None:
    dataset_root = '.'

TEST_ROOT = None
for root, dirs, files in os.walk(dataset_root):
    if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
        TEST_ROOT = root
        break
if TEST_ROOT is None:
    TEST_ROOT = dataset_root

COCO_ANN_PATH = os.path.join(dataset_root, 'test.json')
TRAIN_ANN_PATH = os.path.join(dataset_root, 'train.json')

CFG_PATH = "configs/custom/ip102_t1.py"
if not os.path.exists(CFG_PATH) and os.path.exists("configs/open_world/mowod/custom/ip102_t1.py"):
    CFG_PATH = "configs/open_world/mowod/custom/ip102_t1.py"

KAGGLE_TASK1_CKPT = "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/1/best_coco_Current class AP50_epoch_5.pth"

if os.path.exists(KAGGLE_TASK1_CKPT):
    CKPT_PATH = KAGGLE_TASK1_CKPT
else:
    ckpt_candidates = (
        glob.glob('/kaggle/input/**/best_*.pth', recursive=True) +
        glob.glob('work_dirs/ip102_t1/best_*.pth') +
        glob.glob('work_dirs/ip102_t1/*.pth')
    )
    if ckpt_candidates:
        CKPT_PATH = ckpt_candidates[0]
    else:
        CKPT_PATH = KAGGLE_TASK1_CKPT

OUT_DIR = "/kaggle/working/OW_OVD/paper_visuals"
os.makedirs(OUT_DIR, exist_ok=True)

print("Dataset Root :", dataset_root)
print("TEST_ROOT    :", TEST_ROOT)
print("COCO_ANN_PATH:", COCO_ANN_PATH)
print("CFG_PATH     :", CFG_PATH)
print("CKPT_PATH    :", CKPT_PATH)
print("OUT_DIR      :", OUT_DIR)


In [ ]:
# Cell 5: Sinh embeddings và cấu hình dữ liệu cần thiết cho IP102
import json
import torch
import numpy as np
import os
import glob

try:
    import transformers.utils.import_utils
    import transformers.utils
    import transformers.modeling_utils
    transformers.utils.import_utils.check_torch_load_is_safe = lambda: None
    transformers.utils.check_torch_load_is_safe = lambda: None
    transformers.modeling_utils.check_torch_load_is_safe = lambda: None
except Exception:
    pass

from transformers import AutoTokenizer, CLIPTextModelWithProjection

os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

train_json_path = TRAIN_ANN_PATH if 'TRAIN_ANN_PATH' in globals() and os.path.exists(TRAIN_ANN_PATH) else os.path.join(dataset_root, 'train.json')
print(f"-> Đang đọc annotations từ: {train_json_path}")

with open(train_json_path, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

categories = sorted(coco_data['categories'], key=lambda x: x['id'])
class_names = [cat['name'] for cat in categories]
num_classes = len(class_names)
print(f"-> Tổng số lớp học: {num_classes}")

class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(model_name)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))

num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')

print("-> Đã hoàn thành khởi tạo embeddings cho dataset mới!")


In [ ]:
# Cell 6: Nạp mô hình OW-OVD và Checkpoint Task 1
import os
import sys
import torch

repo_root = "/kaggle/working/OW_OVD"
if not os.path.exists(repo_root):
    repo_root = os.path.abspath(os.getcwd())

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.chdir(repo_root)

import mmcv
import mmdet
import mmyolo
import yolo_world
import yolo_world.models

from mmdet.apis import init_detector
from mmengine.runner import load_checkpoint
from mmengine.config import Config

if not getattr(torch.load, "_owd_safe_patch", False):
    _orig_load = torch.load
    def _safe_load(*a, **kw): kw.setdefault("weights_only", False); return _orig_load(*a, **kw)
    _safe_load._owd_safe_patch = True
    _safe_load._owd_original   = _orig_load
    torch.load = _safe_load

print(f"\nInitializing model architecture with {CFG_PATH}...")
model = init_detector(CFG_PATH, checkpoint=None, device="cpu")

print(f"\nLoading Task 1 weights from: {CKPT_PATH}")
load_checkpoint(model, CKPT_PATH, map_location="cpu")

if hasattr(model, "bbox_head") and hasattr(model.bbox_head, "select_att"):
    print("\nCalling select_att() to adjust att_embeddings size...")
    model.bbox_head.select_att(per_class=25)

cfg = Config.fromfile(CFG_PATH)
num_known = getattr(cfg, "cur_intro_cls", 7)
if hasattr(cfg, "class_names") and cfg.class_names:
    active_classes = list(cfg.class_names)[:num_known]
else:
    active_classes = [f"pest_{i}" for i in range(num_known)]

model.dataset_meta = {"classes": active_classes}
class_names = active_classes
UNKNOWN_ID  = num_known
model.eval()

print(f"\nModel ready for Task 1! {len(class_names)} known classes loaded: {class_names}")


In [ ]:
# Cell 7: Dự đoán và xuất ảnh trực quan cho 10 mẫu ảnh test (Đã sửa Mapping Unknown)
from mmdet.apis import inference_detector
import json, os, random
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# Ngưỡng lọc độ tự tin (Đặt 0.15 - 0.25 để lọc bớt nhiễu)
SCORE_THR = 0.15
NUM_KNOWN_TASK1 = 7  # Task 1 chỉ có 7 lớp Known (index 0..6)

# 1. Đọc danh sách tên 25 lớp từ train.json để lấy tên tiếng Anh/khoa học
train_ann_path = TRAIN_ANN_PATH if 'TRAIN_ANN_PATH' in globals() and os.path.exists(TRAIN_ANN_PATH) else os.path.join(dataset_root, 'train.json')
with open(train_ann_path, 'r', encoding='utf-8') as f:
    train_coco = json.load(f)
full_categories = sorted(train_coco['categories'], key=lambda x: x['id'])
full_class_names = [c['name'] for c in full_categories]

# Danh sách 7 lớp Known của Task 1
known_class_names = full_class_names[:NUM_KNOWN_TASK1]

# 2. Lấy danh sách ảnh test mẫu từ COCO test.json
ann_test_path = COCO_ANN_PATH if 'COCO_ANN_PATH' in globals() and os.path.exists(COCO_ANN_PATH) else os.path.join(dataset_root, 'test.json')
img_root_path = TEST_ROOT if 'TEST_ROOT' in globals() and os.path.exists(TEST_ROOT) else dataset_root
out_dir_path  = OUT_DIR if 'OUT_DIR' in globals() else '/kaggle/working/OW_OVD/paper_visuals'
os.makedirs(out_dir_path, exist_ok=True)

with open(ann_test_path, 'r', encoding='utf-8') as f:
    coco = json.load(f)

id2fname = {img['id']: img['file_name'] for img in coco['images']}
annotated_ids = list({ann['image_id'] for ann in coco['annotations']})
random.seed(42)
random.shuffle(annotated_ids)

IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

def find_image(file_name):
    direct = os.path.join(img_root_path, file_name)
    if os.path.exists(direct): return direct
    basename = os.path.basename(file_name)
    for root, dirs, files in os.walk(img_root_path):
        if basename in files: return os.path.join(root, basename)
    stem = os.path.splitext(basename)[0]
    for ext in IMAGE_EXTS:
        hits = glob.glob(os.path.join(img_root_path, '**', stem + ext), recursive=True)
        if hits: return hits[0]
    return None

img2anns = {}
for ann in coco['annotations']:
    img2anns.setdefault(ann['image_id'], []).append(ann)

cat_id2name = {c['id']: c['name'] for c in coco['categories']}

sample_paths = []
sample_gt_labels = []

for img_id in annotated_ids:
    if len(sample_paths) >= 10: break
    fname = id2fname[img_id]
    path  = find_image(fname)
    if path is None: continue
    anns = img2anns.get(img_id, [])
    if not anns: continue
    sample_paths.append(path)
    sample_gt_labels.append([cat_id2name.get(a['category_id'], '?') for a in anns])

fname2gt = {}
for img in coco['images']:
    anns = img2anns.get(img['id'], [])
    bn = os.path.basename(img['file_name'])
    gt_labels = list({cat_id2name.get(a['category_id'], '?') for a in anns})
    fname2gt[bn] = gt_labels

try:
    font_sm = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 18)
except Exception:
    font_sm = ImageFont.load_default()

model.eval()
print(f'-> Đang chạy suy luận Task 1 (Known <= 6, Unknown >= 7)...')

for idx, img_path in enumerate(sample_paths, start=1):
    result = inference_detector(model, img_path)
    preds  = result.pred_instances
    scores = preds.scores.cpu().numpy()
    labels = preds.labels.cpu().numpy().astype(int)

    # Lọc theo SCORE_THR
    valid_mask = scores >= SCORE_THR
    scores = scores[valid_mask]
    labels = labels[valid_mask]

    img  = Image.open(img_path).convert('RGB')
    W, H = img.width, img.height
    basename = os.path.basename(img_path)
    gt_labels = fname2gt.get(basename, [])

    if len(scores) > 0:
        top_idx   = int(np.argmax(scores))
        top_score = float(scores[top_idx])
        top_lid   = int(labels[top_idx])
        
        # BẢN CHẤT OW-OVD TASK 1: Mọi class index >= 7 đều là UNKNOWN!
        is_unknown = top_lid >= NUM_KNOWN_TASK1
        if is_unknown:
            pred_name = 'unknown'
        else:
            pred_name = known_class_names[top_lid] if top_lid < len(known_class_names) else f'known_{top_lid}'
    else:
        top_score = 0.0
        pred_name = 'no detection'
        is_unknown = False

    border = 6
    border_color = (255, 50, 50) if is_unknown else (255, 215, 0)
    draw = ImageDraw.Draw(img)
    for b in range(border):
        draw.rectangle([b, b, W-1-b, H-1-b], outline=border_color)

    banner_h = 36
    banner = Image.new('RGB', (W, banner_h), color=(30, 30, 30))
    bdraw  = ImageDraw.Draw(banner)
    pred_txt = f'PRED: {pred_name} ({top_score:.2f})'
    bdraw.text((8, 6), pred_txt, fill=border_color, font=font_sm)

    gt_banner = Image.new('RGB', (W, 30), color=(20, 60, 20))
    gdraw     = ImageDraw.Draw(gt_banner)
    gt_txt    = 'GT: ' + ', '.join(gt_labels) if gt_labels else 'GT: (none)'
    gdraw.text((8, 5), gt_txt, fill=(100, 255, 100), font=font_sm)

    total_h = banner_h + H + 30
    canvas  = Image.new('RGB', (W, total_h), color=(20, 20, 20))
    canvas.paste(banner, (0, 0))
    canvas.paste(img,    (0, banner_h))
    canvas.paste(gt_banner, (0, banner_h + H))

    out_path = os.path.join(out_dir_path, f'pred_{idx:02d}.png')
    canvas.save(out_path)
    status = 'UNKNOWN' if is_unknown else 'KNOWN'
    print(f'Image {idx:02d}: {basename} | PRED: {pred_name} ({top_score:.2f}) | GT: {gt_labels} [{status}]')

print(f'\n-> Đã hoàn thành! Kết quả trực quan được lưu tại: {out_dir_path}')
print('Viền vàng = Lớp đã học ở Task 1 (index 0..6), Viền đỏ = Unknown (index >= 7)')
